# Mass Roads Label Graph Overlay

This notebook loops through all label masks in:

`/home/ri/Desktop/Projects/Datasets/Mass_Roads/dataset/train/label/`

For each label:
1. Loads and binarizes the label mask
2. Extracts a graph with `mask2graph.extract_graph`
3. Plots the label and graph overlay together

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path
import sys
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'mask2graph').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mask2graph import load_run_config, run_experiment

CONFIG_PATH = ROOT / 'configs' / 'retinal_augmentation_demo.yaml'
cfg = load_run_config(CONFIG_PATH)
print('config:', CONFIG_PATH)
print('input:', cfg.input.path)
print('run dir:', cfg.output.run_dir)

from mask2graph import ExtractConfig, extract_graph

config: /home/ri/Desktop/Projects/mask2graph/configs/retinal_augmentation_demo.yaml
input: /home/ri/Desktop/Projects/mask2graph/notebooks/data/retinal_binary_mask.gif
run dir: /home/ri/Desktop/Projects/mask2graph/runs/retinal_augmentation_demo


In [2]:
label_dir = Path('/home/ri/Desktop/Projects/Datasets/ProcessedDatasets/MassRoads/train/labels')
label_paths = sorted(label_dir.glob('*.npy'))

print(f'Label directory: {label_dir}')
print(f'Total label files: {len(label_paths)}')
print('First 5 files:', [p.name for p in label_paths[:5]])

Label directory: /home/ri/Desktop/Projects/Datasets/ProcessedDatasets/MassRoads/train/labels
Total label files: 1071
First 5 files: ['10078660_15.npy', '10078675_15.npy', '10078690_15.npy', '10078705_15.npy', '10078720_15.npy']


In [3]:
# Optional: adjust config if needed
config = ExtractConfig()

# Conservative mask cleanup (pixel units because spacing isn't passed in this notebook)
config.cleanup.min_object_size = 9.0
config.cleanup.max_hole_size = 36.0
config.cleanup.max_hole_radius = 2.0

# Junction stabilization and iterative graph normalization (conservative defaults)

# stronger junction stabilization + graph cleanup
config.normalize.junction_dilation_iters = 5      # was 1
config.normalize.prune_spurs_below = 10.0         # was 8
config.normalize.min_component_length = 30.0      # was 25
config.normalize.min_cycle_length = 18.0          # was 12
config.normalize.max_cycle_area = 24.0            # was 16
config.normalize.cycle_length_to_radius_ratio = 5.0  # was 8 (stricter)
config.normalize.contract_short_edges_below = 8.0 # was 2
config.normalize.normalization_max_iter = 20      # was 8

# If you want to test quickly first, set a number (e.g. 20). Keep as None for all labels.
# max_files = None
max_files = 2

# Fast overlay mode: keep full graph geometry and skip optional heavyweight checks.
config.geometry.compute_curvature = False
config.simplify.method = 'none'
config.validation.enabled = False
config.validation.validate_coverage = False
config.validation.validate_topology = False
config.validation.validate_embedding = False

# Plot/display only the first few overlays, while still processing up to max_files.
PLOT_RESULTS = True
PLOT_SAMPLE_COUNT = 2
SAVE_FIGURES = True
FIGURE_DIR = ROOT / 'runs' / 'mass_roads_overlays'

In [4]:
from matplotlib.collections import LineCollection


def load_binary_label(path: Path) -> np.ndarray:
    arr = np.load(path, allow_pickle=False)
    if arr.ndim > 2:
        arr = arr[..., 0]
    return (arr != 0)


def plot_graph_overlay(mask: np.ndarray, graph, title: str, save_path: Path | None = None) -> None:
    fig, ax = plt.subplots(figsize=(16, 16))
    ax.imshow(mask, cmap='gray', interpolation='nearest')

    edge_paths = [np.asarray(edge.path_xyz[:, :2], dtype=np.float64) for edge in graph.edges if len(edge.path_xyz) >= 2]
    if edge_paths:
        ax.add_collection(LineCollection(edge_paths, colors='cyan', linewidths=1.0, alpha=0.85))

    if graph.nodes:
        node_xy = np.asarray([[n.xyz[0], n.xyz[1]] for n in graph.nodes], dtype=np.float64)
        ax.scatter(node_xy[:, 0], node_xy[:, 1], s=50, c='red', marker='o', alpha=0.9, zorder=3)

    ax.set_title(title)
    ax.set_axis_off()
    ax.set_xlim(-0.5, mask.shape[1] - 0.5)
    ax.set_ylim(mask.shape[0] - 0.5, -0.5)
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved figure: {save_path}')
    plt.show()
    plt.close(fig)

In [5]:
selected_paths = label_paths if max_files is None else label_paths[:max_files]
print(f'Processing {len(selected_paths)} label files...')

failed = []
total_nodes = 0
total_edges = 0

for i, path in enumerate(selected_paths, start=1):
    try:
        mask = load_binary_label(path)
        graph = extract_graph(mask, config=config, return_debug=False)

        total_nodes += len(graph.nodes)
        total_edges += len(graph.edges)

        title = f"[{i}/{len(selected_paths)}] {path.name} | nodes={len(graph.nodes)} edges={len(graph.edges)}"
        if PLOT_RESULTS and i <= PLOT_SAMPLE_COUNT:
            save_path = FIGURE_DIR / f'{path.stem}_overlay.png' if SAVE_FIGURES else None
            plot_graph_overlay(mask, graph, title, save_path=save_path)
    except Exception as exc:  # keep loop going even if one file fails
        failed.append((path.name, str(exc)))
        print(f"FAILED: {path.name} -> {exc}")

ok_count = len(selected_paths) - len(failed)
print(f"Done. Failed files: {len(failed)}")
if ok_count > 0:
    print(f"Graph totals -> avg_nodes={total_nodes / ok_count:.1f}, avg_edges={total_edges / ok_count:.1f}")
if failed:
    print('First 10 failures:')
    for name, err in failed[:10]:
        print(f"- {name}: {err}")

Processing 2 label files...


KeyboardInterrupt: 